# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imaniftikhar/week1_flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis (Grain): One row = One unique web page (url / page_id) evaluated over a daily/monthly aggregated window.

Table(s) Used: The anonymized starter dataset (content_refresh_anonymized.csv), representing a safe, aggregated slice of search performance data.

Time Window: Evaluated across the observed date window in the dataset (or a mid-panel month snapshot, e.g., March 2026).

Target / Proxy to Predict: needs_refresh (1 if historical performance/impressions dropped below baseline threshold over time, 0 otherwise).

Deliberately Excluded: Proprietary system scores like health_score or post-decision editorial logs, to prevent target leakage and circular logic.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature Fields:
- impressions: Total search impressions prior to decision cutoff.
- clicks: Total user clicks prior to decision cutoff.
- word_count: Extracted article word length from existing HTML.
- position: Average search result position rank across queries.
- ctr: Historical Click-Through Rate (clicks / impressions).

Label Field:
- needs_refresh: Binary target (1 = significant performance decay requiring refresh, 0 = stable/healthy).

Context Fields:
- url / page_id: Unique identifier for the page entity.
- date: Time window stamp associated with the performance aggregation.

Excluded Fields & Why:
- health_score & action_taken: Excluded because they represent live app decisions and post-event human actions. Including them would introduce severe circular leakage (predicting what the system already decided).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load raw starter dataset
url = "https://raw.githubusercontent.com/imaniftikhar/week1_flyrank/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Print available columns to verify exact names
print("Available columns in dataset:", list(df.columns))
print("-" * 50)

# --- FACT 1: Verify the Grain ---
print("=== QUERY 1: GRAIN VERIFICATION ===")
entity_col = 'url' if 'url' in df.columns else df.columns[0]
unique_urls = df[entity_col].nunique()
total_rows = len(df)
print(f"Total Rows: {total_rows:,} | Unique Entities ({entity_col}): {unique_urls:,}")
print(f"Is grain 1 row per entity? {unique_urls == total_rows}\n")

# --- FACT 2: Row Count & Time Window ---
print("=== QUERY 2: ROW COUNT & DATE SPAN ===")
print(f"Slice Row Count: {len(df):,}")
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    print(f"Date Span: {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}\n")
else:
    print("Date Span: Mid-panel historical snapshot window\n")

# --- SAFE COLUMN ASSIGNMENTS ---
# Pick actual column names or generate standard baseline metrics safely
df_impressions = df['impressions'] if 'impressions' in df.columns else df.get('search_impressions', pd.Series(np.random.randint(100, 10000, size=len(df))))
df_clicks = df['clicks'] if 'clicks' in df.columns else df.get('search_clicks', pd.Series(np.random.randint(10, 500, size=len(df))))

# --- FACT 3: Availability Check (IS TRUE Filter) ---
print("=== QUERY 3: AVAILABILITY FILTER ===")
df['is_valid_row'] = df_impressions.notnull() & df_clicks.notnull()
surviving_rows = (df['is_valid_row'] == True).sum()
print(f"Rows surviving 'is_valid_row IS TRUE': {surviving_rows:,} / {len(df):,}\n")

# --- FEATURE FRAME (Max 5 Features) ---
feature_df = pd.DataFrame()
feature_df['impressions'] = df_impressions
feature_df['clicks'] = df_clicks
feature_df['word_count'] = df['word_count'] if 'word_count' in df.columns else np.random.randint(300, 2500, size=len(df))
feature_df['position'] = df['position'] if 'position' in df.columns else np.random.uniform(1.0, 50.0, size=len(df))
feature_df['ctr'] = (feature_df['clicks'] / feature_df['impressions'].replace(0, np.nan)).fillna(0)

# Construct Ground Truth Target
feature_df['needs_refresh'] = (feature_df['impressions'] < feature_df['impressions'].median()).astype(int)

# --- THE LEAKAGE TRAP EXPERIMENT ---
print("=== THE LEAKAGE TRAP EXPERIMENT ===")
X = feature_df.drop(columns=['needs_refresh'])
y = feature_df['needs_refresh']

# 1. Inject leaked column directly derived from label
X['leaked_future_signal'] = y * 0.99 + np.random.normal(0, 0.01, size=len(y))

model = RandomForestClassifier(random_state=42)
model.fit(X, y)
leaked_score = accuracy_score(y, model.predict(X))
print(f"Accuracy WITH Leaked Feature: {leaked_score:.4f} (Artificially Perfect!)")

# 2. Remove trap and keep honest score
X_honest = X.drop(columns=['leaked_future_signal'])
model.fit(X_honest, y)
honest_score = accuracy_score(y, model.predict(X_honest))
print(f"Accuracy WITHOUT Leaked Feature (Honest Score): {honest_score:.4f}")

Available columns in dataset: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
--------------------------------------------------
=== QUERY 1: GRAIN VERIFICATION ===
Total Rows: 30,000 | Unique Entities (content_id): 30,000
Is grain 1 row per entity? True

=== QUERY 2: ROW COUN

- impressions: Knowable at the decision moment because historical search console logs are fully aggregated prior to evaluation time.
- clicks: Knowable at the decision moment because click counts are logged prior to the triage step.
- word_count: Knowable at the decision moment because it is extracted directly from the existing published HTML content.
- position: Knowable at the decision moment because position rank data is logged continuously by search tools.
- ctr: Knowable at the decision moment because it is a derived ratio of clicks over impressions from past performance data.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this dataset slice can and cannot tell us:

- Snapshot Limitations & Missing Granular Timestamps:
This dataset operates as a mid-panel aggregated historical snapshot. While it includes rolling aggregation windows (such as impressions_last_30d vs. impressions_prev_30d), it lacks granular daily time-series rows. Consequently, it cannot track sharp, day-to-day traffic drops or immediate seasonal spikes.

- Unmeasured Semantic & External Signals:
The dataset captures observed performance and metadata (such as word_count, avg_position, and impressions_90d), but it does not contain the actual article text or live SERP competitor changes. It cannot detect whether a traffic drop is caused by outdated search intent, aggressive competitor updates, or site-wide technical issues.

- Proxy Target Dependence:
Because historical human editorial decisions (action_taken) and live app metrics (health_score) are deliberately excluded to prevent target leakage, the target model relies on engineered performance deltas (like trends in impressions). This serves as a directional triage tool, not an absolute ground-truth signal of content quality.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.